[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/30_cosine_lr_solution.ipynb)

# 🟡 Solution: Cosine LR Scheduler with Warmup

*Training · Medium*

Reference implementation. Try it yourself in `30_cosine_lr.ipynb` first.

---
Implement the learning-rate schedule essentially every modern transformer is
trained with: **linear warmup, then cosine decay**.

$$
\eta(t) =
\begin{cases}
\eta_{\max}\dfrac{t}{T_w} & t < T_w \\[2ex]
\eta_{\min} + \tfrac12(\eta_{\max}-\eta_{\min})
\left(1 + \cos\left(\pi\dfrac{t-T_w}{T-T_w}\right)\right) & t \ge T_w
\end{cases}
$$

### Signature
```python
def cosine_lr_schedule(step, total_steps, warmup_steps, max_lr, min_lr=0.0):
    ...
```

Note the argument order — `total_steps` comes **before** `warmup_steps` and
`max_lr`.

### Rules
- Do not use `optax.warmup_cosine_decay_schedule`
- `step` may be a scalar **or an array** of steps, so branch with `jnp.where`,
  not a Python `if`
- Must be `jax.jit`-able with `step` traced
- Past `total_steps` the rate stays clamped at `min_lr`

### Boundary conventions this is graded on
- $\eta(0) = 0$
- $\eta(T_w) = \eta_{\max}$ exactly — warmup ends *at* the peak
- $\eta(T) = \eta_{\min}$ exactly
- the halfway point of decay is $(\eta_{\max}+\eta_{\min})/2$

### Why warmup exists
At step 0 Adam's second-moment estimate has seen exactly one gradient, so
$\hat{m}/\sqrt{\hat{v}}$ is an unreliable direction with magnitude pinned near 1.
Taking full-size steps in a badly-estimated direction is how early training
diverges, and the deeper the network the worse it gets. Warmup buys the moment
estimates time to become meaningful.

Cosine decay then matters at the other end: it holds a high rate for a long
time and anneals smoothly to near zero, which beats step decay empirically and,
unlike a linear ramp, does not waste the final steps at a rate too small to
make progress.

### A JAX note
The PyTorch original branches with a Python `if` and returns a float, which is
fine for a scalar step. Writing it with `jnp.where` instead costs nothing, and
buys you a schedule that works on a whole array of steps at once and survives
`jit` when the step is traced.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def cosine_lr_schedule(step, total_steps, warmup_steps, max_lr, min_lr=0.0):
    step = jnp.asarray(step, dtype=jnp.float32)

    warmup_lr = max_lr * step / jnp.maximum(warmup_steps, 1)

    # Clip so the schedule flattens at min_lr instead of turning back upward
    # once step runs past total_steps.
    decay_span = jnp.maximum(total_steps - warmup_steps, 1)
    progress = jnp.clip((step - warmup_steps) / decay_span, 0.0, 1.0)
    cosine_lr = min_lr + 0.5 * (max_lr - min_lr) * (1 + jnp.cos(jnp.pi * progress))

    # jnp.where, not a Python if — keeps this vectorised and jittable.
    return jnp.where(step < warmup_steps, warmup_lr, cosine_lr)

In [ ]:
# 🔍 Verify
import jax.numpy as jnp

steps = jnp.arange(0, 1001, 100)
lrs = cosine_lr_schedule(steps, total_steps=1000, warmup_steps=100, max_lr=1e-3)
for s, lr in zip(steps.tolist(), lrs.tolist()):
    bar = "█" * int(lr / 1e-3 * 40)
    print(f"{s:>5}  {lr:.6f}  {bar}")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("cosine_lr")